##### HIGH LEVEL IMPLEMENTATION OF THE PAPER 'Neural Machine Translation by Jointly Learning to Align and Translate (Bahdanau et al., 2014) ' USING PYTORCH


here is the link to the paper: https://arxiv.org/abs/1409.0473

things to try:

- remove padding zeros

- setting bias to False/True on the Attention model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import re
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.nn.functional as F
import random
import time
import math

#### if you are using google colab , uncomment the next section to download the data

In [ ]:
# if you are using google colab , uncomment this section to download the data
# import urllib.request
# import zipfile
# import os

# # Download a small, clean dataset of English-German pairs
# url = "https://www.manythings.org/anki/deu-eng.zip"
# zip_path = "deu-eng.zip"

# # Create a request with a User-Agent header to avoid HTTP Error 406
# req = urllib.request.Request(
#     url,
#     headers={
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
#     },
# )

# # Open the URL and retrieve the data
# with urllib.request.urlopen(req) as response:
#     with open(zip_path, 'wb') as out_file:
#         out_file.write(response.read())

# # Extract it
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(".")

In [ ]:
with open("../deu.txt", "r", encoding="utf-8") as f:
    lines = f.read().strip().split("\n")
# Keep a subset of 10,000 short sentences to save memory and train fast
pairs = []
for line in lines[:3]:
    # The file is tab-separated: English \t German \t Attribution
    parts = line.split("\n")[0].split("\t")
    pairs.append((parts[0], parts[1]))

print(f"Sample pair: {pairs[:3]}")

In [ ]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.num_words = 4

    def add_sentence(self, sentence):
        sentence = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", sentence.lower())
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.num_words
                self.idx2word[self.num_words] = word
                self.num_words += 1

    def numericalize(self, sentence):
        sentence = re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", sentence.lower())
        return [
            self.word2idx.get(word, self.word2idx["<UNK>"]) for word in sentence.split()
        ]


# Initialize separate vocabularies for Source and Target languages
eng_vocab = Vocabulary()
deu_vocab = Vocabulary()

for eng, deu in pairs:
    eng_vocab.add_sentence(eng)
    deu_vocab.add_sentence(deu)

In [ ]:
type(pairs[0])

In [ ]:
print(pairs)

In [ ]:
print(eng_vocab)

In [ ]:
print(eng_vocab.word2idx)
print(eng_vocab.idx2word)

In [ ]:
print(deu_vocab.word2idx)
print(deu_vocab.idx2word)

In [ ]:
print(eng_vocab.numericalize("hi, go out "))

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        src_sent, trg_sent = self.pairs[idx]
        # Numericalize text strings
        src_tokens = self.src_vocab.numericalize(src_sent)
        trg_tokens = self.trg_vocab.numericalize(trg_sent)

        # src_tokens = src_tokens[::-1]
        src_tensor = torch.tensor(
            src_tokens + [self.src_vocab.word2idx["<EOS>"]], dtype=torch.long
        )

        trg_tensor = torch.tensor(
            [self.trg_vocab.word2idx["<SOS>"]]
            + trg_tokens
            + [self.trg_vocab.word2idx["<EOS>"]],
            dtype=torch.long,
        )
        return src_tensor, trg_tensor


In [ ]:
# Collate function to dynamically pad batches to equivalent lengths
def collate_fn(batch):

    src_batch, trg_batch = zip(*batch)

    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)

    return src_batch, trg_batch


In [ ]:
dataset = TranslationDataset(pairs, eng_vocab, deu_vocab)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [ ]:
ss = torch.tensor([19, 99, 483, 132, 1275, 3], dtype=torch.long)
print(ss, ss.shape)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)

        # Using PyTorch's native deep multi-layer LSTM cell wrapper
        self.rnn = nn.LSTM(
            emb_dim, hidden_dim, n_layers, batch_first=True, bidirectional=True
        )

    def forward(self, src):

        embedded = self.embedding(src)

        all_outputs, (hidden, cell) = self.rnn(embedded)

        return all_outputs


In [ ]:
ss = torch.tensor(
    [
        [[1, 2], [3, 4], [5, 6]],
        [[7, 8], [9, 10], [4, 2]],
        [[1, 8], [6, 4], [0, 4]],
        [[1, 2], [3, 5], [8, 9]],
    ],
    dtype=torch.float,
)
print(ss, "\n", ss.shape)

In [ ]:
ss = ss.view(2, 2, 3, 2)


In [ ]:
print(ss, "\n", ss.shape)

In [ ]:
(
    [
        [
            [
                [1, 2],
                [3, 4],
                [5, 6],
            ],
            [
                [7, 8],
                [9, 10],
                [4, 2],
            ],
        ],
        [
            [
                [1, 8],
                [6, 4],
                [0, 4],
            ],
            [
                [1, 2],
                [3, 5],
                [8, 9],
            ],
        ],
    ]
)

In [ ]:
fc_hidden = nn.Linear(2 * 2, 2)


In [ ]:
hidden_layers, cell_layers = [], []
for layer in range(2):
    h_cat = torch.cat((ss[layer, 0], ss[layer, 1]), dim=1)
    print(h_cat, h_cat.shape)
    t_h = torch.tanh(fc_hidden(h_cat))
    print("tt: ", t_h.shape)
    hidden_layers.append(t_h)

    # (n_layers, batch, hidden_dim) - matches what the decoder's LSTM expects


In [ ]:
hidden = torch.stack(hidden_layers, dim=0)

In [ ]:
print(
    hidden_layers,
)
print(hidden, hidden.shape)

In [ ]:
[
    ([[-0.9998, -0.9924], [-1.0000, -0.9994], [0.1346, -0.9716]],),
    ([[0.9890, 0.0890], [-0.9619, -0.9958], [-0.9997, -0.9891]],),
]

(
    [
        [[-0.9998, -0.9924], [-1.0000, -0.9994], [0.1346, -0.9716]],
        [[0.9890, 0.0890], [-0.9619, -0.9958], [-0.9997, -0.9891]],
    ],
)

In [ ]:
ss = torch.tensor(
    [
        [1, 2, 5, 6],
        [3, 4, 1, 0],
        [5, 6, 7, 3],
    ],
    dtype=torch.float,
)

dd = torch.tensor(
    [
        [3, 1, 1, 2],
        [5, 0, 5, 8],
        [7, 9, 9, 0],
    ],
    dtype=torch.float,
)
print(ss, "\n", ss.shape)

In [ ]:
n = torch.stack([ss, dd], dim=0)
print(n, n.shape)

In [ ]:
n = torch.stack([n, n], dim=0)
print(n, n.shape)

In [ ]:
#
class AlignmentModel(nn.Module):
    def __init__(self, decoder_hidden_dim, encoder_ouput_dim, attention_neurons_dim):
        super().__init__()
        self.Wa = nn.Linear(decoder_hidden_dim, attention_neurons_dim)
        self.Ua = nn.Linear(encoder_ouput_dim, attention_neurons_dim)
        self.Va = nn.Linear(attention_neurons_dim, 1)

    def forward(self, encoder_ouput, decoder_hidden):
        U_a = self.Ua(encoder_ouput)
        W_a = self.Wa(decoder_hidden).unsqueeze(1)
        tan_out = torch.tanh(W_a + U_a)
        V_a = self.Va(tan_out)
        attention_weights = F.softmax(V_a, dim=1)
        context_vector = torch.bmm(attention_weights.permute(0, 2, 1), encoder_ouput)
        return context_vector, attention_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_dim,
        emb_dim,
        encoder_ouput_dim,
        decoder_hidden_dim,
        attention_neurons_dim,
        n_layers,
    ):
        super().__init__()

        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = AlignmentModel(
            encoder_ouput_dim, decoder_hidden_dim, attention_neurons_dim
        )

        lstm_input_dim = emb_dim + (encoder_ouput_dim * 2)
        self.rnn = nn.LSTM(
            lstm_input_dim, decoder_hidden_dim, n_layers, batch_first=True
        )

        self.maxout_linear = nn.Linear(
            decoder_hidden_dim + emb_dim + (encoder_ouput_dim * 2),
            decoder_hidden_dim * 2,
        )
        self.fc_out = nn.Linear(decoder_hidden_dim, output_dim)

    def forward(
        self,
        # prev_memory,
        # prev_word,
        encoder_outputs,
        input_token,
        hidden,
        cell,
    ):

        # Input token shape: (batch_size) -> unsqueeze to track sequence steps (batch_size, 1)
        input_token = input_token.unsqueeze(1)

        embedded = self.embedding(input_token)
        context_vector, attention_weights = self.attention(hidden, encoder_outputs)

        lstm_input = torch.cat((embedded, context_vector), dim=2)
        # Process the single sequence step using preceding context records
        all_output, (hidden, cell) = self.rnn(lstm_input, (hidden, cell))

        context_vector = context_vector.squeeze(1)
        embedded = embedded.squeeze(1)

        # 3. Maxout Activation Logic (Splitting tensor in half and taking element-wise max)
        combined_features = torch.cat((hidden, embedded, context_vector), dim=1)
        linear_project = self.maxout_linear(combined_features)

        # Split into two equal feature chunks along the columns
        t1, t2 = torch.chunk(linear_project, 2, dim=1)
        maxout_output = torch.max(t1, t2)  # [batch_size, decoder_hidden_dim]

        # 4. Predict Vocab Probabilities
        prediction = self.fc_out(maxout_output)  # [batch_size, vocab_size]

        return prediction, hidden, cell, attention_weights

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):

        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features
        # Tensor vector space shell to accumulate individual sequence element predictions
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        # Capture fixed vector representations from the Encoder segment
        hidden, cell = self.encoder(src)

        # Initialize output loop step with the sequence starting boundary signal
        input_token = trg[:, 0]
        for t in range(1, trg_len):
            prediction, hidden, cell = self.decoder(input_token, hidden, cell)

            outputs[:, t, :] = prediction

            # Determine teacher forcing strategy for the next input step
            teacher_force = random.random() < teacher_forcing_ratio

            top1 = prediction.argmax(1)
            input_token = trg[:, t] if teacher_force else top1

        return outputs

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scale dimensions way down relative to the 1000-cell, 4-layer paper parameters to fit your Colab notebook
INPUT_DIM = eng_vocab.num_words
OUTPUT_DIM = deu_vocab.num_words
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HIDDEN_DIM = 256
N_LAYERS = 2

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM, N_LAYERS)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM, N_LAYERS)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)  # Ignore <PAD> tokens in loss calculation

print(f"Model initialized successfully. Device: {device}")

In [ ]:
def train(model, dataloader, optimizer, criterion, clip):

    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(dataloader):
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()
        # Forward pass: pass data through the model layout
        output = model(src, trg)

        # trg = [batch size, trg len]
        # output = [batch size, trg len, output dim]

        output_dim = output.shape[-1]
        # Reshape output and target tensors to compute cross-entropy loss
        output = output[:, 1:].reshape(-1, output_dim)

        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        # Gradient clipping - a standard trick for training stable LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


# Run a quick training cycle test
N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    start_time = time.time()
    train_loss = train(model, dataloader, optimizer, criterion, CLIP)
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )

In [ ]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()  # Turn off dropout / batch norm layers

    # 1. Preprocess and reverse the input sentence exactly like training
    tokens = src_vocab.numericalize(sentence)
    # tokens = tokens[::-1]  # Secret Sauce: Reverse it!
    tokens = tokens + [src_vocab.word2idx["<EOS>"]]

    # 2. Convert to tensor and add a batch dimension (shape: [1, seq_len])
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
    print("src", src_tensor)
    # 3. Pass through Encoder to get Context Vectors
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    # 4. Initialize the loop with <SOS>
    trg_indexes = [trg_vocab.word2idx["<SOS>"]]

    # 5. Decode step-by-step
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        print("trg index", trg_tensor)
        with torch.no_grad():
            prediction, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        # Get the model's highest scoring prediction
        pred_token = prediction.argmax(1).item()
        trg_indexes.append(pred_token)
        print(trg_indexes)
        # Stop if the model predicts the End of Sentence token
        if pred_token == trg_vocab.word2idx["<EOS>"]:
            break

    # 6. Convert integer indices back into actual German words
    trg_tokens = [trg_vocab.idx2word[i] for i in trg_indexes]

    # Return string while cutting out the structural control symbols
    return " ".join(trg_tokens[1:-1])


# --- TEST IT OUT ---
example_input = "Hi, go away"
translation = translate_sentence(example_input, eng_vocab, deu_vocab, model, device)
print(f"English: {example_input}")
print(f"German Translation: {translation}")